In [ ]:
import numpy as np
from scipy import signal
import soundfile as sf
import matplotlib.pyplot as plt
import os
!wget -q https://raw.githubusercontent.com/OmerOzbektas/LTI-Acoustic-Convolution/main/audio_inputs/my_voice.wav
!wget -q https://raw.githubusercontent.com/OmerOzbektas/LTI-Acoustic-Convolution/main/impulse_responses/cathedral_ir.wav

def plot_signals(u, h, y, fs, output_path):
    t_u = np.linspace(0, len(u) / fs, num=len(u))
    t_h = np.linspace(0, len(h) / fs, num=len(h))
    t_y = np.linspace(0, len(y) / fs, num=len(y))

    fig, axs = plt.subplots(3, 1, figsize=(10, 8))
    fig.tight_layout(pad=4.0)

    axs[0].plot(t_u, u, color='blue')
    axs[0].set_title('1. Input Signal (Dry Audio)')
    axs[0].set_ylabel('Amplitude')

    axs[1].plot(t_h, h, color='red')
    axs[1].set_title('2. Impulse Response (System Characteristic)')
    axs[1].set_ylabel('Amplitude')

    axs[2].plot(t_y, y, color='purple')
    axs[2].set_title('3. Output Signal (Convolved Audio)')
    axs[2].set_xlabel('Time (Seconds)')
    axs[2].set_ylabel('Amplitude')

    plt.savefig(output_path)
    print(f"[+] Waveform graphs saved to: {output_path}")
    plt.close()

def apply_convolution(input_audio_path, impulse_response_path, output_audio_path):
    try:
        u, fs_u = sf.read(input_audio_path)
        h, fs_h = sf.read(impulse_response_path)
    except Exception as e:
        print(f"[!] Error: Could not read the files. Details: {e}")
        return

    print(f"Input Signal Sampling Rate (fs): {fs_u} Hz")
    print(f"Impulse Response Sampling Rate (fs): {fs_h} Hz")

    if fs_u != fs_h:
        print("\n[!] WARNING: Sampling rates do not match! Resulting audio might be distorted.")

    if len(u.shape) > 1:
        u = np.mean(u, axis=1)
    if len(h.shape) > 1:
        h = np.mean(h, axis=1)

    y = signal.convolve(u, h, mode='full')
    y_normalized = y / np.max(np.abs(y))

    output_dir = os.path.dirname(output_audio_path)
    if output_dir:
         os.makedirs(output_dir, exist_ok=True)

    sf.write(output_audio_path, y_normalized, fs_u)
    print(f"\n[+] SUCCESS! Output audio saved to: {output_audio_path}")

    plot_path = os.path.join(output_dir if output_dir else ".", "waveform_comparison.png")
    plot_signals(u, h, y_normalized, fs_u, plot_path)

input_audio = "my_voice.wav"
ir_audio = "St.PaulCathedral.wav"
output_audio = "result_audio.wav"

apply_convolution(input_audio, ir_audio, output_audio)

Input Signal Sampling Rate (fs): 48000 Hz
Impulse Response Sampling Rate (fs): 44100 Hz

[!] WARNING: Sampling rates do not match! Resulting audio might be distorted.

[+] SUCCESS! Output audio saved to: result_audio.wav
[+] Waveform graphs saved to: ./waveform_comparison.png
